In [89]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
# Exploratory Data Analysis (EDA)
from sklearn.preprocessing import OneHotEncoder

from sklearn.preprocessing import StandardScaler

## Task info

Your task

Implement a machine learning system that predicts being diagnosed with heart disease

1) Apply EDA to extract useful insights
2) Build your preprocessing pipeline consisting of scaling, label-encoding, one-hot encoding
3) Try the 3 models we covered logistic regression, svm & knn
 
Evaluate your model accuracy, precesion, recall, ......

Kindly deploy on a streamlit app

In [90]:
os.listdir('data/feature eng data/')

['feature_eng_data.csv']

In [91]:
df = pd.read_csv('data/feature eng data/feature_eng_data.csv')

# Data info
- -EDA-1 = 
- A /data/raw/Exam_Score_Prediction.csv 
- B data/feature eng data/feature_eng_data.csv

- -Pre-processed-2
- A data/feature eng data/feature_eng_data.csv
- B data/preprocessed data/pre-processed_data.csv

- -Machine-learning-3 = 
- A /data/preprocessed data/pre-processed_data.csv
- B /artifacts/svm_pipeline.pkl

### 1. Label Encoding (Ordinal)
- Wenn die drei Werte eine logische Rangfolge haben (z. B. klein < mittel < gross oder normal < leicht defekt < schwer defekt), nutzt du Label Encoding. Dabei werden Zahlen wie 0, 1, 2 vergeben.
Warum? Das Modell lernt, dass 2 "mehr" oder "stärker" ist als 0.

In [92]:
df.shape

(1025, 14)

## Suche für Label Encoding

In [93]:
df["chest_pain_type"].unique()

# Label Encoding

array(['typical angina', 'atypical angina', 'non-anginal pain',
       'asymptomatic'], dtype=object)

In [94]:
# Profi-Tipp: Zeigt die prozentuale Verteilung (0.45 = 45%)
print(df['chest_pain_type'].value_counts(normalize=True) * 100)

chest_pain_type
typical angina      48.487805
non-anginal pain    27.707317
atypical angina     16.292683
asymptomatic         7.512195
Name: proportion, dtype: float64


# Label encoding

In [95]:
def encode_chest_pain_type ( df ):
    df['chest_pain_type'] = df['chest_pain_type'].map({

        'asymptomatic' : 0, 'atypical angina': 1,
        'non-anginal pain': 2, 'typical angina': 3
    })
    return df

df = encode_chest_pain_type( df )

In [96]:
df.sample(4)

,age,sex,chest_pain_type,resisting_blood_pressure,cholesterol_level,fasting_blood_sugar,rest_ecg,max_heart_rate_achieved,exercise_induced_angina,st_depression,st_slope,num_major_vessels,thalassemia,diagnosis
591,63,female,3,108,269,lower than 120mg/ml,ST-T wave abnormality,169,yes,1.8,flat,2,fixed defect,Diagnosed
748,44,male,2,120,226,lower than 120mg/ml,ST-T wave abnormality,169,no,0.0,downsloping,0,fixed defect,Un-Diagnosed
780,44,male,3,120,169,lower than 120mg/ml,ST-T wave abnormality,144,yes,2.8,upsloping,0,normal,Diagnosed
258,38,male,0,120,231,lower than 120mg/ml,ST-T wave abnormality,182,yes,3.8,flat,0,reversable defect,Diagnosed


### Hier ist null falsch

In [97]:
df["thalassemia"].unique()

# Null '0' ist falsch muss bereinigt werden

array(['reversable defect', 'fixed defect', 'normal'], dtype=object)

In [98]:
df["thalassemia"].value_counts()

thalassemia
fixed defect         548
reversable defect    413
normal                64
Name: count, dtype: int64

## Datenbereinigung:

- Fall 1 (ID 14): Der 52-jährige Mann
Status: "Diagnosed" (Herzkrankheit liegt vor).
Meine Entscheidung: reversable defect.

- Fall 2 (ID 319): Die 53-jährige Frau
Status: "Un-Diagnosed" (Gesund).
Meine Entscheidung: normal

In [99]:
df [ df["thalassemia"] == '0']

,age,sex,chest_pain_type,resisting_blood_pressure,cholesterol_level,fasting_blood_sugar,rest_ecg,max_heart_rate_achieved,exercise_induced_angina,st_depression,st_slope,num_major_vessels,thalassemia,diagnosis


In [100]:
df.loc[14, 'thalassemia'] = 'fixed defect'
df.loc[[14]]

,age,sex,chest_pain_type,resisting_blood_pressure,cholesterol_level,fasting_blood_sugar,rest_ecg,max_heart_rate_achieved,exercise_induced_angina,st_depression,st_slope,num_major_vessels,thalassemia,diagnosis
14,52,male,3,128,204,higher than 120mg/ml,ST-T wave abnormality,156,yes,1.0,flat,0,fixed defect,Diagnosed


In [101]:
print(df.loc[14, "thalassemia"])

fixed defect


In [102]:
df.loc[319, 'thalassemia'] = 'reversable defect'

In [103]:
df.loc[[319]]

,age,sex,chest_pain_type,resisting_blood_pressure,cholesterol_level,fasting_blood_sugar,rest_ecg,max_heart_rate_achieved,exercise_induced_angina,st_depression,st_slope,num_major_vessels,thalassemia,diagnosis
319,53,female,2,128,216,lower than 120mg/ml,normal,115,no,0.0,downsloping,0,reversable defect,Un-Diagnosed


# Label encoding

In [104]:
df["thalassemia"].unique()

# Label eccoding

array(['reversable defect', 'fixed defect', 'normal'], dtype=object)

In [105]:
def encode_thalassemia ( df ):
    df["thalassemia"] = df["thalassemia"].map({
        'normal': 0, 'fixed defect': 1,
        'reversable defect': 2
    })
    return df
df = encode_thalassemia ( df )

In [106]:
df["diagnosis"].unique()

# One Hot encoding

array(['Diagnosed', 'Un-Diagnosed'], dtype=object)

In [107]:
df.sample(3)

,age,sex,chest_pain_type,resisting_blood_pressure,cholesterol_level,fasting_blood_sugar,rest_ecg,max_heart_rate_achieved,exercise_induced_angina,st_depression,st_slope,num_major_vessels,thalassemia,diagnosis
782,64,female,3,130,303,lower than 120mg/ml,ST-T wave abnormality,122,no,2.0,flat,2,1,Un-Diagnosed
787,51,male,3,140,298,lower than 120mg/ml,ST-T wave abnormality,122,yes,4.2,flat,3,2,Diagnosed
203,64,male,0,170,227,lower than 120mg/ml,normal,155,no,0.6,flat,0,2,Un-Diagnosed


# One Hot label

In [108]:
df["sex"].unique()

# One Hot Label Encoding   

array(['male', 'female'], dtype=object)

In [109]:
df["fasting_blood_sugar"].unique()

# One Hot Label Encoding             

array(['lower than 120mg/ml', 'higher than 120mg/ml'], dtype=object)

In [110]:
df["rest_ecg"].unique()
   
# one hot Encoding          

array(['ST-T wave abnormality', 'normal', 'left ventricular hypertrophy'],
      dtype=object)

In [111]:
df["exercise_induced_angina"].unique()
   
# One Hot Label Encoding

array(['no', 'yes'], dtype=object)

In [112]:
df["st_slope"].unique()

# Label Encoding

array(['downsloping', 'upsloping', 'flat'], dtype=object)

In [113]:
df["diagnosis"].unique()

# One Hot encoding

array(['Diagnosed', 'Un-Diagnosed'], dtype=object)

In [114]:
df.columns

Index(['age', 'sex', 'chest_pain_type', 'resisting_blood_pressure',
       'cholesterol_level', 'fasting_blood_sugar', 'rest_ecg',
       'max_heart_rate_achieved', 'exercise_induced_angina', 'st_depression',
       'st_slope', 'num_major_vessels', 'thalassemia', 'diagnosis'],
      dtype='object')

# Columns to one-hot-encoder:
- 'fasting_blood_sugar', 'rest_ecg', 'diagnosis', 'st_slope', 'exercise_induced_angina' 
- Die Spalte **diagnosis** draussen nehmen, weil  dein Zielwert (Label) ist.

In [115]:
def apply_1_hot_encoding ( df ):
    df = pd.get_dummies( df,
                        columns= [ 'sex','fasting_blood_sugar', 'rest_ecg', 'st_slope', 'exercise_induced_angina' ],
                         dtype= int )
    return df

- Die Spalte **diagnosis** draussen nehmen, weil  dein Zielwert (Label) ist.

In [116]:
df = pd.get_dummies (df,
                     columns = ['sex','fasting_blood_sugar', 'rest_ecg', 'st_slope', 'exercise_induced_angina' ],
                     dtype= int)
df

,age,chest_pain_type,resisting_blood_pressure,cholesterol_level,max_heart_rate_achieved,st_depression,num_major_vessels,thalassemia,diagnosis,sex_female,...,fasting_blood_sugar_higher than 120mg/ml,fasting_blood_sugar_lower than 120mg/ml,rest_ecg_ST-T wave abnormality,rest_ecg_left ventricular hypertrophy,rest_ecg_normal,st_slope_downsloping,st_slope_flat,st_slope_upsloping,exercise_induced_angina_no,exercise_induced_angina_yes
0,52,3,125,212,168,1.0,2,2,Diagnosed,0,...,0,1,1,0,0,1,0,0,1,0
1,53,3,140,203,155,3.1,0,2,Diagnosed,0,...,1,0,0,0,1,0,0,1,0,1
2,70,3,145,174,125,2.6,0,2,Diagnosed,0,...,0,1,1,0,0,0,0,1,0,1
3,61,3,148,203,161,0.0,1,2,Diagnosed,0,...,0,1,1,0,0,1,0,0,1,0
4,62,3,138,294,106,1.9,3,1,Diagnosed,1,...,1,0,1,0,0,0,1,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1020,59,1,140,221,164,0.0,0,1,Un-Diagnosed,0,...,0,1,1,0,0,1,0,0,0,1
1021,60,3,125,258,141,2.8,1,2,Diagnosed,0,...,0,1,0,0,1,0,1,0,0,1
1022,47,3,110,275,118,1.0,1,1,Diagnosed,0,...,0,1,0,0,1,0,1,0,0,1
1023,50,3,110,254,159,0.0,0,1,Un-Diagnosed,1,...,0,1,0,0,1,1,0,0,1,0


In [117]:
df.shape

(1025, 21)

In [118]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1025 entries, 0 to 1024
Data columns (total 21 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   age                                       1025 non-null   int64  
 1   chest_pain_type                           1025 non-null   int64  
 2   resisting_blood_pressure                  1025 non-null   int64  
 3   cholesterol_level                         1025 non-null   int64  
 4   max_heart_rate_achieved                   1025 non-null   int64  
 5   st_depression                             1025 non-null   float64
 6   num_major_vessels                         1025 non-null   int64  
 7   thalassemia                               1025 non-null   int64  
 8   diagnosis                                 1025 non-null   object 
 9   sex_female                                1025 non-null   int64  
 10  sex_male                            

Diese Funktion sollte für am Schluss wichtig sein.
- Die Spalte **diagnosis** draussen nehmen, weil  dein Zielwert (Label) ist.

In [119]:
def apply_1_hot_encoding(df, full_df):
    for col in [ 'sex','fasting_blood_sugar', 'rest_ecg',  'st_slope', 'exercise_induced_angina' ]:
        df[col] = pd.Categorical(df[col], categories=full_df[col].unique())

        df = pd.get_dummies(df,
                            columns= ['sex','fasting_blood_sugar', 'rest_ecg', 'st_slope', 'exercise_induced_angina' ],
                            dtype=int)
        return df

In [120]:
one_hot_encoder = OneHotEncoder()
one_hot_encoder

,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values within a single feature, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute... versionadded:: 0.20",'auto'
,"drop drop: {'first', 'if_binary'} or an array-like of shape (n_features,), default=NoneSpecifies a methodology to use to drop one of the categories perfeature. This is useful in situations where perfectly collinearfeatures cause problems, such as when feeding the resulting datainto an unregularized linear regression model.However, dropping one category breaks the symmetry of the originalrepresentation and can therefore induce a bias in downstream models,for instance for penalized linear classification or regression models.- None : retain all features (the default).- 'first' : drop the first category in each feature. If only one category is present, the feature will be dropped entirely.- 'if_binary' : drop the first category in each feature with two categories. Features with 1 or more than 2 categories are left intact.- array : ``drop[i]`` is the category in feature ``X[:, i]`` that should be dropped.When `max_categories` or `min_frequency` is configured to groupinfrequent categories, the dropping behavior is handled after thegrouping... versionadded:: 0.21 The parameter `drop` was added in 0.21... versionchanged:: 0.23 The option `drop='if_binary'` was added in 0.23... versionchanged:: 1.1 Support for dropping infrequent categories.",None
,"sparse_output sparse_output: bool, default=TrueWhen ``True``, it returns a :class:`scipy.sparse.csr_matrix`,i.e. a sparse matrix in ""Compressed Sparse Row"" (CSR) format... versionadded:: 1.2 `sparse` was renamed to `sparse_output`",True
,"dtype dtype: number type, default=np.float64Desired dtype of output.",<class 'numpy.float64'>
,"handle_unknown handle_unknown: {'error', 'ignore', 'infrequent_if_exist', 'warn'}, default='error'Specifies the way unknown categories are handled during :meth:`transform`.- 'error' : Raise an error if an unknown category is present during transform.- 'ignore' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. In the inverse transform, an unknown category will be denoted as None.- 'infrequent_if_exist' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will map to the infrequent category if it exists. The infrequent category will be mapped to the last position in the encoding. During inverse transform, an unknown category will be mapped to the category denoted `'infrequent'` if it exists. If the `'infrequent'` category does not exist, then :meth:`transform` and :meth:`inverse_transform` will handle an unknown category as with `handle_unknown='ignore'`. Infrequent categories exist based on `min_frequency` and `max_categories`. Read more in the :ref:`User Guide `.- 'warn' : When an unknown category is encountered during transform a warning is issued, and the encoding then proceeds as described for `handle_unknown=""infrequent_if_exist""`... versionchanged:: 1.1 `'infrequent_if_exist'` was added to automatically handle unknown categories and infrequent categories... versionadded:: 1.6 The option `""warn""` was added in 1.6.",'error'
,"min_frequency min_frequency: int or float, default=NoneSpecifies the minimum frequency below which a category will beconsidered infrequent.- If `int`, categories with a smaller cardinality will be considered infrequent.- If `float`, categories with a smaller cardinality than `min_frequency * n_samples` will be considered infrequent... versionadded:: 1.1 Read more in the :ref:`User Guide `.",None
,"max_catego

In [121]:
data = df.head(3)
data

,age,chest_pain_type,resisting_blood_pressure,cholesterol_level,max_heart_rate_achieved,st_depression,num_major_vessels,thalassemia,diagnosis,sex_female,...,fasting_blood_sugar_higher than 120mg/ml,fasting_blood_sugar_lower than 120mg/ml,rest_ecg_ST-T wave abnormality,rest_ecg_left ventricular hypertrophy,rest_ecg_normal,st_slope_downsloping,st_slope_flat,st_slope_upsloping,exercise_induced_angina_no,exercise_induced_angina_yes
0,52,3,125,212,168,1.0,2,2,Diagnosed,0,...,0,1,1,0,0,1,0,0,1,0
1,53,3,140,203,155,3.1,0,2,Diagnosed,0,...,1,0,0,0,1,0,0,1,0,1
2,70,3,145,174,125,2.6,0,2,Diagnosed,0,...,0,1,1,0,0,0,0,1,0,1


In [122]:
one_hot_encoder.fit( data )

,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values within a single feature, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute... versionadded:: 0.20",'auto'
,"drop drop: {'first', 'if_binary'} or an array-like of shape (n_features,), default=NoneSpecifies a methodology to use to drop one of the categories perfeature. This is useful in situations where perfectly collinearfeatures cause problems, such as when feeding the resulting datainto an unregularized linear regression model.However, dropping one category breaks the symmetry of the originalrepresentation and can therefore induce a bias in downstream models,for instance for penalized linear classification or regression models.- None : retain all features (the default).- 'first' : drop the first category in each feature. If only one category is present, the feature will be dropped entirely.- 'if_binary' : drop the first category in each feature with two categories. Features with 1 or more than 2 categories are left intact.- array : ``drop[i]`` is the category in feature ``X[:, i]`` that should be dropped.When `max_categories` or `min_frequency` is configured to groupinfrequent categories, the dropping behavior is handled after thegrouping... versionadded:: 0.21 The parameter `drop` was added in 0.21... versionchanged:: 0.23 The option `drop='if_binary'` was added in 0.23... versionchanged:: 1.1 Support for dropping infrequent categories.",None
,"sparse_output sparse_output: bool, default=TrueWhen ``True``, it returns a :class:`scipy.sparse.csr_matrix`,i.e. a sparse matrix in ""Compressed Sparse Row"" (CSR) format... versionadded:: 1.2 `sparse` was renamed to `sparse_output`",True
,"dtype dtype: number type, default=np.float64Desired dtype of output.",<class 'numpy.float64'>
,"handle_unknown handle_unknown: {'error', 'ignore', 'infrequent_if_exist', 'warn'}, default='error'Specifies the way unknown categories are handled during :meth:`transform`.- 'error' : Raise an error if an unknown category is present during transform.- 'ignore' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. In the inverse transform, an unknown category will be denoted as None.- 'infrequent_if_exist' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will map to the infrequent category if it exists. The infrequent category will be mapped to the last position in the encoding. During inverse transform, an unknown category will be mapped to the category denoted `'infrequent'` if it exists. If the `'infrequent'` category does not exist, then :meth:`transform` and :meth:`inverse_transform` will handle an unknown category as with `handle_unknown='ignore'`. Infrequent categories exist based on `min_frequency` and `max_categories`. Read more in the :ref:`User Guide `.- 'warn' : When an unknown category is encountered during transform a warning is issued, and the encoding then proceeds as described for `handle_unknown=""infrequent_if_exist""`... versionchanged:: 1.1 `'infrequent_if_exist'` was added to automatically handle unknown categories and infrequent categories... versionadded:: 1.6 The option `""warn""` was added in 1.6.",'error'
,"min_frequency min_frequency: int or float, default=NoneSpecifies the minimum frequency below which a category will beconsidered infrequent.- If `int`, categories with a smaller cardinality will be considered infrequent.- If `float`, categories with a smaller cardinality than `min_frequency * n_samples` will be considered infrequent... versionadded:: 1.1 Read more in the :ref:`User Guide `.",None
,"max_catego

In [123]:
data

,age,chest_pain_type,resisting_blood_pressure,cholesterol_level,max_heart_rate_achieved,st_depression,num_major_vessels,thalassemia,diagnosis,sex_female,...,fasting_blood_sugar_higher than 120mg/ml,fasting_blood_sugar_lower than 120mg/ml,rest_ecg_ST-T wave abnormality,rest_ecg_left ventricular hypertrophy,rest_ecg_normal,st_slope_downsloping,st_slope_flat,st_slope_upsloping,exercise_induced_angina_no,exercise_induced_angina_yes
0,52,3,125,212,168,1.0,2,2,Diagnosed,0,...,0,1,1,0,0,1,0,0,1,0
1,53,3,140,203,155,3.1,0,2,Diagnosed,0,...,1,0,0,0,1,0,0,1,0,1
2,70,3,145,174,125,2.6,0,2,Diagnosed,0,...,0,1,1,0,0,0,0,1,0,1


In [124]:
one_hot_encoder = OneHotEncoder()

In [125]:
one_hot_encoder.fit_transform ( data )

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 63 stored elements and shape (3, 40)>

In [126]:
df.isna().sum()

age                                         0
chest_pain_type                             0
resisting_blood_pressure                    0
cholesterol_level                           0
max_heart_rate_achieved                     0
st_depression                               0
num_major_vessels                           0
thalassemia                                 0
diagnosis                                   0
sex_female                                  0
sex_male                                    0
fasting_blood_sugar_higher than 120mg/ml    0
fasting_blood_sugar_lower than 120mg/ml     0
rest_ecg_ST-T wave abnormality              0
rest_ecg_left ventricular hypertrophy       0
rest_ecg_normal                             0
st_slope_downsloping                        0
st_slope_flat                               0
st_slope_upsloping                          0
exercise_induced_angina_no                  0
exercise_induced_angina_yes                 0
dtype: int64

In [127]:
df[df["age"].isna()]

,age,chest_pain_type,resisting_blood_pressure,cholesterol_level,max_heart_rate_achieved,st_depression,num_major_vessels,thalassemia,diagnosis,sex_female,...,fasting_blood_sugar_higher than 120mg/ml,fasting_blood_sugar_lower than 120mg/ml,rest_ecg_ST-T wave abnormality,rest_ecg_left ventricular hypertrophy,rest_ecg_normal,st_slope_downsloping,st_slope_flat,st_slope_upsloping,exercise_induced_angina_no,exercise_induced_angina_yes


In [128]:
df.isna().sum()

age                                         0
chest_pain_type                             0
resisting_blood_pressure                    0
cholesterol_level                           0
max_heart_rate_achieved                     0
st_depression                               0
num_major_vessels                           0
thalassemia                                 0
diagnosis                                   0
sex_female                                  0
sex_male                                    0
fasting_blood_sugar_higher than 120mg/ml    0
fasting_blood_sugar_lower than 120mg/ml     0
rest_ecg_ST-T wave abnormality              0
rest_ecg_left ventricular hypertrophy       0
rest_ecg_normal                             0
st_slope_downsloping                        0
st_slope_flat                               0
st_slope_upsloping                          0
exercise_induced_angina_no                  0
exercise_induced_angina_yes                 0
dtype: int64

In [129]:
df = df.drop(index=319).reset_index(drop=True)

In [130]:
os.listdir('data/preprocessed-data/')

['preprocessed-data.csv']

In [131]:
df.to_csv('data/preprocessed-data/preprocessed-data.csv', index= False)